# NOTEBOOK 5: INTEGRACIÓN POWER BI Y EXPORTACIÓN FINAL

---

## Proyecto: Arquitectura de BI y Big Data para Análisis del Turismo Académico en Medellín

**Objetivo del Notebook:** Preparar datos finales en formato óptimo para Power BI, crear tablas dimensionales, calcular KPIs y generar reportes ejecutivos.

---

### Contenido:
1. Carga de datos procesados y resultados de modelos
2. Creación de modelo dimensional (esquema estrella)
3. Cálculo de KPIs principales
4. Exportación de tablas para Power BI
5. Script SQL para PostgreSQL
6. Documentación para Power BI
7. Reporte ejecutivo final

---
## 1. CONFIGURACIÓN E IMPORTACIÓN

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
from datetime import datetime

# Rutas
# Como el notebook está en 'notebooks/', subimos un nivel para llegar a la raíz
BASE_DIR = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
DATA_PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RESULTS_DIR = BASE_DIR / 'data' / 'results'
REPORTS_DIR = BASE_DIR / 'outputs' / 'reportes'
SQL_DIR = BASE_DIR / 'sql'

SQL_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Configuración completada")
print(f"Directorio base: {BASE_DIR}")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 2. CARGA DE DATOS PROCESADOS

In [ ]:
# Cargar datos consolidados limpios
df = pd.read_csv(DATA_PROCESSED_DIR / 'datos_consolidados_limpios.csv')

print("="*80)
print("DATOS CARGADOS PARA POWER BI")
print("="*80)
print(f"\nRegistros: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
print(f"\nUniversidades: {df['UNIVERSIDAD'].value_counts().to_dict()}")

# Cargar proyecciones si existen
try:
    proyecciones = pd.read_csv(RESULTS_DIR / 'proyecciones_impacto_economico.csv')
    print(f"\n✓ Proyecciones cargadas: {len(proyecciones)} periodos futuros")
except:
    proyecciones = None
    print("\n⚠ No se encontraron proyecciones (ejecutar Notebook 4 primero)")

---
## 3. CREACIÓN DE MODELO DIMENSIONAL (ESQUEMA ESTRELLA)

Crearemos tablas dimensionales y una tabla de hechos para optimizar el rendimiento en Power BI.

### 3.1 Dimensión Tiempo

In [ ]:
print("\nCreando Dimensión Tiempo...")

# Crear dimensión tiempo única
dim_tiempo = df[['AÑO', 'SEMESTRE', 'PERIODO']].drop_duplicates().reset_index(drop=True)
dim_tiempo['ID_TIEMPO'] = range(1, len(dim_tiempo) + 1)
dim_tiempo['TRIMESTRE'] = dim_tiempo['SEMESTRE'].apply(lambda x: 1 if x == 1 else 2)
dim_tiempo['ANIO_SEMESTRE'] = dim_tiempo['AÑO'].astype(str) + '-S' + dim_tiempo['SEMESTRE'].astype(str)

# Reordenar columnas
dim_tiempo = dim_tiempo[['ID_TIEMPO', 'AÑO', 'SEMESTRE', 'TRIMESTRE', 'PERIODO', 'ANIO_SEMESTRE']]

print(f"✓ Dimensión Tiempo creada: {len(dim_tiempo)} registros")
display(dim_tiempo.head())

### 3.2 Dimensión Geografía (Países)

In [ ]:
print("\nCreando Dimensión Geografía...")

# Crear dimensión países
dim_geografia = df[['PAIS_EXTRANJERO']].drop_duplicates().reset_index(drop=True)
dim_geografia['ID_PAIS'] = range(1, len(dim_geografia) + 1)

# Clasificar por región (simplificado)
def clasificar_region(pais):
    pais_upper = str(pais).upper()
    if any(p in pais_upper for p in ['MÉXICO', 'ESTADOS UNIDOS', 'BRASIL', 'VENEZUELA']):
        return 'América'
    elif any(p in pais_upper for p in ['ESPAÑA', 'ITALIA', 'GRECIA', 'RUMANIA', 'TURQUÍA']):
        return 'Europa'
    else:
        return 'Otro'

dim_geografia['REGION'] = dim_geografia['PAIS_EXTRANJERO'].apply(clasificar_region)
dim_geografia = dim_geografia[['ID_PAIS', 'PAIS_EXTRANJERO', 'REGION']]

print(f"✓ Dimensión Geografía creada: {len(dim_geografia)} países")
print(f"\nDistribución por región:")
print(dim_geografia['REGION'].value_counts())
display(dim_geografia.head(10))

### 3.3 Dimensión Universidad

In [ ]:
print("\nCreando Dimensión Universidad...")

dim_universidad = pd.DataFrame({
    'ID_UNIVERSIDAD': [1, 2, 3],
    'NOMBRE_UNIVERSIDAD': ['IUSH', 'Universidad de Antioquia', 'UNAC'],
    'TIPO': ['Privada', 'Pública', 'Privada'],
    'CIUDAD': ['Medellín', 'Medellín', 'Medellín']
})

print(f"✓ Dimensión Universidad creada")
display(dim_universidad)

### 3.4 Dimensión Tipo de Movilidad

In [ ]:
print("\nCreando Dimensión Tipo de Movilidad...")

dim_tipo_movilidad = df[['TIPO_MOV_EST_EXTRANJ']].drop_duplicates().reset_index(drop=True)
dim_tipo_movilidad['ID_TIPO_MOVILIDAD'] = range(1, len(dim_tipo_movilidad) + 1)
dim_tipo_movilidad = dim_tipo_movilidad[['ID_TIPO_MOVILIDAD', 'TIPO_MOV_EST_EXTRANJ']]

print(f"✓ Dimensión Tipo Movilidad creada: {len(dim_tipo_movilidad)} tipos")
display(dim_tipo_movilidad)

### 3.5 Tabla de Hechos (Fact Table)

In [ ]:
print("\nCreando Tabla de Hechos...")

# Merge con dimensiones para obtener IDs
fact_movilidad = df.copy()

# Mapeo con dimensión tiempo
fact_movilidad = fact_movilidad.merge(
    dim_tiempo[['ID_TIEMPO', 'AÑO', 'SEMESTRE']], 
    on=['AÑO', 'SEMESTRE'], 
    how='left'
)

# Mapeo con dimensión geografía
fact_movilidad = fact_movilidad.merge(
    dim_geografia[['ID_PAIS', 'PAIS_EXTRANJERO']], 
    on='PAIS_EXTRANJERO', 
    how='left'
)

# Mapeo con dimensión universidad
universidad_map = {'IUSH': 1, 'Universidad de Antioquia': 2, 'UNAC': 3}
fact_movilidad['ID_UNIVERSIDAD'] = fact_movilidad['UNIVERSIDAD'].map(universidad_map)

# Mapeo con dimensión tipo movilidad
fact_movilidad = fact_movilidad.merge(
    dim_tipo_movilidad[['ID_TIPO_MOVILIDAD', 'TIPO_MOV_EST_EXTRANJ']], 
    on='TIPO_MOV_EST_EXTRANJ', 
    how='left'
)

# Seleccionar columnas para la tabla de hechos
fact_movilidad_final = fact_movilidad[[
    'ID_TIEMPO',
    'ID_PAIS',
    'ID_UNIVERSIDAD',
    'ID_TIPO_MOVILIDAD',
    'NUM_DIAS_MOVILIDAD',
    'FINANCIACION_TOTAL',
    'GASTO_ESTIMADO_DIRECTO',
    'GASTO_ESTIMADO_TOTAL'
]].copy()

# Agregar ID único
fact_movilidad_final['ID_HECHO'] = range(1, len(fact_movilidad_final) + 1)

# Reordenar
fact_movilidad_final = fact_movilidad_final[[
    'ID_HECHO', 'ID_TIEMPO', 'ID_PAIS', 'ID_UNIVERSIDAD', 'ID_TIPO_MOVILIDAD',
    'NUM_DIAS_MOVILIDAD', 'FINANCIACION_TOTAL', 'GASTO_ESTIMADO_DIRECTO', 'GASTO_ESTIMADO_TOTAL'
]]

print(f"✓ Tabla de Hechos creada: {len(fact_movilidad_final)} registros")
print(f"\nVista previa:")
display(fact_movilidad_final.head())
print(f"\nInformación:")
print(fact_movilidad_final.info())

---
## 4. CÁLCULO DE KPIs PRINCIPALES

Calcularemos los KPIs clave para el dashboard de Power BI.

In [ ]:
print("\n" + "="*80)
print("CÁLCULO DE KPIs PRINCIPALES")
print("="*80)

# KPIs generales
kpis = {
    'Total_Estudiantes_Internacionales': len(df),
    'Paises_Representados': df['PAIS_EXTRANJERO'].nunique(),
    'Duracion_Promedio_Dias': df['NUM_DIAS_MOVILIDAD'].mean(),
    'Duracion_Mediana_Dias': df['NUM_DIAS_MOVILIDAD'].median(),
    'Impacto_Economico_Total_COP': df['GASTO_ESTIMADO_TOTAL'].sum(),
    'Impacto_Economico_Total_USD': df['GASTO_ESTIMADO_TOTAL'].sum() / 4000,
    'Financiacion_Total_COP': df['FINANCIACION_TOTAL'].sum(),
    'Estudiantes_Por_Universidad_Promedio': len(df) / df['UNIVERSIDAD'].nunique(),
    'Instituciones_Extranjeras': df['INSTITUCION_EXTRANJERA'].nunique(),
    'Pais_Principal': df['PAIS_EXTRANJERO'].value_counts().index[0],
    'Tipo_Movilidad_Principal': df['TIPO_MOV_EST_EXTRANJ'].value_counts().index[0]
}

# KPIs por año (último año disponible)
ultimo_anio = df['AÑO'].max()
df_ultimo_anio = df[df['AÑO'] == ultimo_anio]

kpis_anuales = {
    f'Estudiantes_{ultimo_anio}': len(df_ultimo_anio),
    f'Impacto_Economico_{ultimo_anio}_COP': df_ultimo_anio['GASTO_ESTIMADO_TOTAL'].sum(),
    f'Tasa_Crecimiento_Anual_Pct': 0  # Se calculará con más datos históricos
}

kpis.update(kpis_anuales)

# Crear DataFrame de KPIs
kpis_df = pd.DataFrame(list(kpis.items()), columns=['KPI', 'Valor'])

print("\nKPIs CALCULADOS:")
print("="*80)
display(kpis_df)

# Exportar KPIs
kpis_df.to_csv(RESULTS_DIR / 'kpis_powerbi.csv', index=False)
print(f"\n✓ KPIs exportados: {RESULTS_DIR / 'kpis_powerbi.csv'}")

---
## 5. EXPORTACIÓN DE TABLAS PARA POWER BI

In [ ]:
print("\n" + "="*80)
print("EXPORTANDO TABLAS PARA POWER BI")
print("="*80)

# Exportar dimensiones
dim_tiempo.to_csv(RESULTS_DIR / 'dim_tiempo.csv', index=False)
print(f"\n✓ Dimensión Tiempo exportada ({len(dim_tiempo)} registros)")

dim_geografia.to_csv(RESULTS_DIR / 'dim_geografia.csv', index=False)
print(f"✓ Dimensión Geografía exportada ({len(dim_geografia)} registros)")

dim_universidad.to_csv(RESULTS_DIR / 'dim_universidad.csv', index=False)
print(f"✓ Dimensión Universidad exportada ({len(dim_universidad)} registros)")

dim_tipo_movilidad.to_csv(RESULTS_DIR / 'dim_tipo_movilidad.csv', index=False)
print(f"✓ Dimensión Tipo Movilidad exportada ({len(dim_tipo_movilidad)} registros)")

# Exportar tabla de hechos
fact_movilidad_final.to_csv(RESULTS_DIR / 'fact_movilidad.csv', index=False)
print(f"✓ Tabla de Hechos exportada ({len(fact_movilidad_final)} registros)")

# Exportar datos completos también (para análisis ad-hoc)
df.to_csv(RESULTS_DIR / 'datos_completos_powerbi.csv', index=False)
print(f"✓ Datos completos exportados ({len(df)} registros)")

# Exportar proyecciones si existen
if proyecciones is not None:
    proyecciones.to_csv(RESULTS_DIR / 'proyecciones_futuras.csv', index=False)
    print(f"✓ Proyecciones futuras exportadas ({len(proyecciones)} periodos)")

print("\n" + "="*80)
print("RESUMEN DE ARCHIVOS EXPORTADOS")
print("="*80)
archivos = list(RESULTS_DIR.glob('*.csv'))
for archivo in archivos:
    tamanio_kb = archivo.stat().st_size / 1024
    print(f"  - {archivo.name}: {tamanio_kb:.2f} KB")

---
## 6. GENERACIÓN DE SCRIPT SQL PARA POSTGRESQL

In [ ]:
print("\nGenerando script SQL para PostgreSQL...")

sql_script = '''
-- ============================================================================
-- SCRIPT DE CREACIÓN DE BASE DE DATOS - TURISMO ACADÉMICO MEDELLÍN
-- Proyecto: Arquitectura de BI y Big Data
-- Autor: Proyecto de Grado - Especialización
-- Fecha: ''' + datetime.now().strftime('%Y-%m-%d') + '''
-- ============================================================================

-- Crear base de datos
CREATE DATABASE turismo_academico_medellin;
\c turismo_academico_medellin;

-- ============================================================================
-- TABLAS DIMENSIONALES
-- ============================================================================

-- Dimensión Tiempo
CREATE TABLE dim_tiempo (
    id_tiempo SERIAL PRIMARY KEY,
    anio INTEGER NOT NULL,
    semestre INTEGER NOT NULL CHECK (semestre IN (1, 2)),
    trimestre INTEGER NOT NULL CHECK (trimestre IN (1, 2)),
    periodo VARCHAR(20),
    anio_semestre VARCHAR(20),
    UNIQUE(anio, semestre)
);

-- Dimensión Geografía
CREATE TABLE dim_geografia (
    id_pais SERIAL PRIMARY KEY,
    pais_extranjero VARCHAR(100) NOT NULL UNIQUE,
    region VARCHAR(50) NOT NULL
);

-- Dimensión Universidad
CREATE TABLE dim_universidad (
    id_universidad SERIAL PRIMARY KEY,
    nombre_universidad VARCHAR(100) NOT NULL UNIQUE,
    tipo VARCHAR(20) CHECK (tipo IN ('Pública', 'Privada')),
    ciudad VARCHAR(50) NOT NULL
);

-- Dimensión Tipo de Movilidad
CREATE TABLE dim_tipo_movilidad (
    id_tipo_movilidad SERIAL PRIMARY KEY,
    tipo_mov_est_extranj VARCHAR(100) NOT NULL UNIQUE
);

-- ============================================================================
-- TABLA DE HECHOS
-- ============================================================================

CREATE TABLE fact_movilidad (
    id_hecho SERIAL PRIMARY KEY,
    id_tiempo INTEGER REFERENCES dim_tiempo(id_tiempo),
    id_pais INTEGER REFERENCES dim_geografia(id_pais),
    id_universidad INTEGER REFERENCES dim_universidad(id_universidad),
    id_tipo_movilidad INTEGER REFERENCES dim_tipo_movilidad(id_tipo_movilidad),
    num_dias_movilidad INTEGER,
    financiacion_total NUMERIC(15, 2),
    gasto_estimado_directo NUMERIC(15, 2),
    gasto_estimado_total NUMERIC(15, 2),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- ============================================================================
-- ÍNDICES PARA OPTIMIZACIÓN
-- ============================================================================

CREATE INDEX idx_fact_tiempo ON fact_movilidad(id_tiempo);
CREATE INDEX idx_fact_pais ON fact_movilidad(id_pais);
CREATE INDEX idx_fact_universidad ON fact_movilidad(id_universidad);
CREATE INDEX idx_fact_tipo_mov ON fact_movilidad(id_tipo_movilidad);

-- ============================================================================
-- VISTAS PARA POWER BI
-- ============================================================================

-- Vista consolidada para análisis
CREATE VIEW vw_movilidad_completa AS
SELECT 
    f.id_hecho,
    t.anio,
    t.semestre,
    t.anio_semestre,
    g.pais_extranjero,
    g.region,
    u.nombre_universidad,
    u.tipo AS tipo_universidad,
    tm.tipo_mov_est_extranj,
    f.num_dias_movilidad,
    f.financiacion_total,
    f.gasto_estimado_directo,
    f.gasto_estimado_total
FROM fact_movilidad f
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
JOIN dim_geografia g ON f.id_pais = g.id_pais
JOIN dim_universidad u ON f.id_universidad = u.id_universidad
JOIN dim_tipo_movilidad tm ON f.id_tipo_movilidad = tm.id_tipo_movilidad;

-- Vista de KPIs
CREATE VIEW vw_kpis_resumen AS
SELECT 
    COUNT(*) AS total_estudiantes,
    COUNT(DISTINCT f.id_pais) AS paises_representados,
    AVG(f.num_dias_movilidad) AS duracion_promedio_dias,
    SUM(f.gasto_estimado_total) AS impacto_economico_total,
    SUM(f.financiacion_total) AS financiacion_total
FROM fact_movilidad f;

-- ============================================================================
-- COMENTARIOS EN LAS TABLAS
-- ============================================================================

COMMENT ON TABLE fact_movilidad IS 'Tabla de hechos con registros de movilidad estudiantil internacional';
COMMENT ON TABLE dim_tiempo IS 'Dimensión temporal con años y semestres';
COMMENT ON TABLE dim_geografia IS 'Dimensión geográfica con países y regiones';
COMMENT ON TABLE dim_universidad IS 'Dimensión de universidades de Medellín';
COMMENT ON TABLE dim_tipo_movilidad IS 'Dimensión de tipos de movilidad académica';

-- ============================================================================
-- FIN DEL SCRIPT
-- ============================================================================

-- Para cargar datos desde CSV usar:
-- \copy dim_tiempo FROM 'dim_tiempo.csv' DELIMITER ',' CSV HEADER;
-- \copy dim_geografia FROM 'dim_geografia.csv' DELIMITER ',' CSV HEADER;
-- \copy dim_universidad FROM 'dim_universidad.csv' DELIMITER ',' CSV HEADER;
-- \copy dim_tipo_movilidad FROM 'dim_tipo_movilidad.csv' DELIMITER ',' CSV HEADER;
-- \copy fact_movilidad FROM 'fact_movilidad.csv' DELIMITER ',' CSV HEADER;
'''

# Guardar script SQL
with open(SQL_DIR / 'crear_tablas_postgresql.sql', 'w', encoding='utf-8') as f:
    f.write(sql_script)

print(f"\n✓ Script SQL generado: {SQL_DIR / 'crear_tablas_postgresql.sql'}")
print("\nEl script incluye:")
print("  - Creación de base de datos")
print("  - Tablas dimensionales (4)")
print("  - Tabla de hechos (1)")
print("  - Índices de optimización")
print("  - Vistas para análisis")
print("  - Comentarios y documentación")

---
## 7. DOCUMENTACIÓN PARA POWER BI

In [ ]:
print("\nGenerando documentación para Power BI...")

documentacion_pbi = f'''
{'='*80}
GUÍA DE INTEGRACIÓN CON POWER BI
{'='*80}

Proyecto: Arquitectura de BI y Big Data - Turismo Académico Medellín
Fecha: {datetime.now().strftime('%Y-%m-%d')}

{'='*80}
1. ARCHIVOS DISPONIBLES PARA IMPORTACIÓN
{'='*80}

MODELO DIMENSIONAL (Recomendado):
  ✓ dim_tiempo.csv - Dimensión temporal
  ✓ dim_geografia.csv - Dimensión de países
  ✓ dim_universidad.csv - Dimensión de universidades
  ✓ dim_tipo_movilidad.csv - Dimensión de tipos de movilidad
  ✓ fact_movilidad.csv - Tabla de hechos principal

DATOS COMPLETOS (Alternativa):
  ✓ datos_completos_powerbi.csv - Dataset consolidado completo

ANÁLISIS Y PROYECCIONES:
  ✓ kpis_powerbi.csv - KPIs principales calculados
  ✓ proyecciones_futuras.csv - Proyecciones de flujos futuros
  ✓ proyecciones_arima.csv - Proyecciones del modelo ARIMA

{'='*80}
2. ESTRUCTURA DEL MODELO DE DATOS
{'='*80}

Esquema Estrella (Star Schema):

                    dim_tiempo
                        |
                        |
  dim_geografia ----  FACT_MOVILIDAD  ---- dim_tipo_movilidad
                        |
                        |
                   dim_universidad

RELACIONES:
  - fact_movilidad[ID_TIEMPO] -> dim_tiempo[ID_TIEMPO] (Many-to-One)
  - fact_movilidad[ID_PAIS] -> dim_geografia[ID_PAIS] (Many-to-One)
  - fact_movilidad[ID_UNIVERSIDAD] -> dim_universidad[ID_UNIVERSIDAD] (Many-to-One)
  - fact_movilidad[ID_TIPO_MOVILIDAD] -> dim_tipo_movilidad[ID_TIPO_MOVILIDAD] (Many-to-One)

{'='*80}
3. MÉTRICAS PRINCIPALES (DAX)
{'='*80}

Total Estudiantes = COUNTROWS(fact_movilidad)

Impacto Económico Total = SUM(fact_movilidad[GASTO_ESTIMADO_TOTAL])

Duración Promedio = AVERAGE(fact_movilidad[NUM_DIAS_MOVILIDAD])

Tasa de Crecimiento = 
VAR AnioActual = CALCULATE([Total Estudiantes], LASTDATE(dim_tiempo[AÑO]))
VAR AnioAnterior = CALCULATE([Total Estudiantes], PREVIOUSYEAR(dim_tiempo[AÑO]))
RETURN
DIVIDE(AnioActual - AnioAnterior, AnioAnterior, 0)

Países Únicos = DISTINCTCOUNT(dim_geografia[PAIS_EXTRANJERO])

{'='*80}
4. VISUALIZACIONES RECOMENDADAS
{'='*80}

PÁGINA 1: RESUMEN EJECUTIVO
  - Tarjetas KPI: Total estudiantes, Impacto económico, Países, Duración promedio
  - Gráfico de línea: Evolución temporal de movilidad
  - Gráfico de barras: Top 10 países de origen
  - Mapa: Distribución geográfica de estudiantes

PÁGINA 2: ANÁLISIS TEMPORAL
  - Gráfico de área: Tendencia anual de estudiantes
  - Gráfico de columnas: Comparación por semestre
  - Tabla: Métricas por periodo
  - Indicador: Tasa de crecimiento

PÁGINA 3: ANÁLISIS GEOGRÁFICO
  - Treemap: Distribución por país
  - Gráfico de barras horizontal: Top países por impacto económico
  - Gráfico circular: Distribución por región
  - Tabla: Detalle por país

PÁGINA 4: ANÁLISIS POR UNIVERSIDAD
  - Gráfico de barras: Estudiantes por universidad
  - Gráfico de columnas agrupadas: Comparación temporal
  - Tabla matriz: Universidad x País

PÁGINA 5: PROYECCIONES
  - Gráfico de línea con proyección: Tendencia futura
  - Tarjetas: Proyecciones para próximos periodos
  - Gráfico de área: Impacto económico proyectado

{'='*80}
5. FILTROS Y SLICERS RECOMENDADOS
{'='*80}

  - Año (dim_tiempo[AÑO])
  - Semestre (dim_tiempo[SEMESTRE])
  - País (dim_geografia[PAIS_EXTRANJERO])
  - Región (dim_geografia[REGION])
  - Universidad (dim_universidad[NOMBRE_UNIVERSIDAD])
  - Tipo de Movilidad (dim_tipo_movilidad[TIPO_MOV_EST_EXTRANJ])

{'='*80}
6. PASOS PARA IMPORTAR EN POWER BI
{'='*80}

OPCIÓN 1: DESDE ARCHIVOS CSV
1. Abrir Power BI Desktop
2. Inicio > Obtener datos > Texto/CSV
3. Seleccionar cada archivo CSV de la carpeta data/results/
4. Transformar datos si es necesario (Power Query)
5. Cargar datos al modelo
6. Crear relaciones en Vista de Modelo:
   - Arrastrar ID_TIEMPO desde fact_movilidad a dim_tiempo
   - Arrastrar ID_PAIS desde fact_movilidad a dim_geografia
   - Arrastrar ID_UNIVERSIDAD desde fact_movilidad a dim_universidad
   - Arrastrar ID_TIPO_MOVILIDAD desde fact_movilidad a dim_tipo_movilidad

OPCIÓN 2: DESDE POSTGRESQL (Recomendado para producción)
1. Ejecutar script SQL: sql/crear_tablas_postgresql.sql
2. Cargar datos CSV en PostgreSQL usando \copy commands
3. En Power BI: Obtener datos > Base de datos > PostgreSQL
4. Conectar a la base de datos turismo_academico_medellin
5. Seleccionar tablas dim_* y fact_movilidad
6. Las relaciones se detectarán automáticamente

{'='*80}
7. CONFIGURACIÓN DE ACTUALIZACIÓN
{'='*80}

Para mantener el dashboard actualizado:
  - Si usa CSV: Re-ejecutar notebooks 1-5 y actualizar datos en Power BI
  - Si usa PostgreSQL: Configurar actualización programada en Power BI Service
  - Frecuencia recomendada: Semestral (coincidiendo con ciclos académicos)

{'='*80}
8. CONTACTO Y SOPORTE
{'='*80}

Para dudas sobre la implementación consultar:
  - Notebooks en carpeta notebooks/
  - Reportes en carpeta outputs/reportes/
  - Documentación técnica del proyecto

{'='*80}
FIN DE LA DOCUMENTACIÓN
{'='*80}
'''

# Guardar documentación
with open(REPORTS_DIR / 'guia_powerbi.txt', 'w', encoding='utf-8') as f:
    f.write(documentacion_pbi)

print(documentacion_pbi)
print(f"\n✓ Documentación guardada: {REPORTS_DIR / 'guia_powerbi.txt'}")

---
## 8. REPORTE EJECUTIVO FINAL

In [ ]:
print("\nGenerando Reporte Ejecutivo Final...")

reporte_ejecutivo = f'''
{'='*80}
REPORTE EJECUTIVO FINAL
Arquitectura de BI y Big Data para el Análisis del Impacto Económico
del Turismo Académico en Medellín
{'='*80}

Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Autor: Proyecto de Grado - Especialización

{'='*80}
1. RESUMEN EJECUTIVO
{'='*80}

Este proyecto desarrolló una arquitectura completa de Inteligencia de Negocios
y Big Data para medir el impacto económico del turismo académico en Medellín,
consolidando datos de movilidad estudiantil internacional de tres universidades.

UNIVERSIDADES ANALIZADAS:
  - IUSH (Institución Universitaria Salazar y Herrera)
  - Universidad de Antioquia
  - UNAC (Universidad Autónoma de las Américas)

{'='*80}
2. RESULTADOS PRINCIPALES
{'='*80}

MÉTRICAS CLAVE:
  Total de estudiantes internacionales: {len(df):,}
  Países representados: {df['PAIS_EXTRANJERO'].nunique()}
  Duración promedio de estadía: {df['NUM_DIAS_MOVILIDAD'].mean():.1f} días
  
IMPACTO ECONÓMICO ESTIMADO:
  Impacto directo: ${df['GASTO_ESTIMADO_DIRECTO'].sum():,.0f} COP
  Impacto total (con multiplicador): ${df['GASTO_ESTIMADO_TOTAL'].sum():,.0f} COP
  Equivalente en USD: ${df['GASTO_ESTIMADO_TOTAL'].sum()/4000:,.0f} USD

TOP 5 PAÍSES DE ORIGEN:
'''

# Agregar top 5 países
for i, (pais, cantidad) in enumerate(df['PAIS_EXTRANJERO'].value_counts().head(5).items(), 1):
    porcentaje = (cantidad / len(df)) * 100
    reporte_ejecutivo += f"  {i}. {pais}: {cantidad} estudiantes ({porcentaje:.1f}%)\n"

reporte_ejecutivo += f'''
{'='*80}
3. PROCESO TÉCNICO EJECUTADO
{'='*80}

FASE 1: EXTRACCIÓN Y EXPLORACIÓN
  ✓ Carga de datos de 3 universidades
  ✓ Identificación de problemas de calidad
  ✓ Exploración inicial de patrones

FASE 2: LIMPIEZA Y TRANSFORMACIÓN (ETL)
  ✓ Eliminación de {len(pd.read_csv(DATA_RAW_DIR / 'iush.csv')) - len(df):,} filas vacías
  ✓ Estandarización de nomenclatura de países
  ✓ Creación de variables derivadas
  ✓ Validación de calidad post-limpieza

FASE 3: ANÁLISIS DESCRIPTIVO
  ✓ Análisis temporal de tendencias
  ✓ Distribución geográfica de estudiantes
  ✓ Análisis por tipo de movilidad
  ✓ Análisis financiero y correlaciones

FASE 4: MODELOS PREDICTIVOS
  ✓ Modelo ARIMA para proyecciones temporales
  ✓ Modelo de Regresión Lineal
  ✓ Modelo Random Forest (comparación)
  ✓ Estimación de impacto económico futuro

FASE 5: INTEGRACIÓN CON POWER BI
  ✓ Creación de modelo dimensional (esquema estrella)
  ✓ Cálculo de KPIs principales
  ✓ Exportación de tablas optimizadas
  ✓ Generación de script SQL para PostgreSQL

{'='*80}
4. ENTREGABLES DEL PROYECTO
{'='*80}

NOTEBOOKS JUPYTER (Documentación completa):
  1. 01_carga_exploracion_datos.ipynb
  2. 02_limpieza_estandarizacion_etl.ipynb
  3. 03_analisis_descriptivo.ipynb
  4. 04_modelos_predictivos.ipynb
  5. 05_integracion_powerbi_exportacion.ipynb

DATOS PROCESADOS:
  - Modelo dimensional (5 tablas CSV)
  - KPIs calculados
  - Proyecciones futuras
  - Dataset consolidado completo

VISUALIZACIONES:
  - {len(list(OUTPUTS_DIR.glob('*.png')))} gráficas de análisis generadas
  - Guardadas en: outputs/graficas/

BASE DE DATOS:
  - Script SQL completo para PostgreSQL
  - Incluye tablas, índices y vistas

DOCUMENTACIÓN:
  - Guía de integración con Power BI
  - Reportes de calidad de datos
  - Reporte de modelos predictivos

{'='*80}
5. RECOMENDACIONES
{'='*80}

CORTO PLAZO:
  1. Implementar dashboard en Power BI siguiendo la guía proporcionada
  2. Validar proyecciones con datos reales cuando estén disponibles
  3. Consolidar datos de Universidad de Antioquia y UNAC

MEDIANO PLAZO:
  1. Automatizar proceso ETL para actualización semestral
  2. Implementar base de datos PostgreSQL en servidor
  3. Ampliar análisis con datos socioeconómicos adicionales

LARGO PLAZO:
  1. Expandir análisis a otras ciudades de Colombia
  2. Integrar con sistemas de gestión universitaria
  3. Desarrollar API para consultas en tiempo real

{'='*80}
6. CONCLUSIONES
{'='*80}

✓ Se logró consolidar exitosamente datos fragmentados de movilidad estudiantil
✓ Se implementó un pipeline ETL automatizado y reproducible
✓ Se cuantificó el impacto económico del turismo académico en Medellín
✓ Se crearon modelos predictivos con métricas de precisión validadas
✓ Se preparó infraestructura completa para dashboards ejecutivos

El sistema desarrollado permite a las universidades y autoridades locales
tomar decisiones basadas en datos sobre políticas de internacionalización
y promoción del turismo académico.

{'='*80}
PROYECTO COMPLETADO EXITOSAMENTE
{'='*80}

Para ejecutar el pipeline completo:
  1. Instalar dependencias: pip install -r requirements.txt
  2. Ejecutar notebooks en orden (01 -> 05)
  3. Importar datos en Power BI usando guia_powerbi.txt

Todos los archivos están organizados y listos para uso en producción.
'''

# Guardar reporte ejecutivo
with open(REPORTS_DIR / 'reporte_ejecutivo_final.md', 'w', encoding='utf-8') as f:
    f.write(reporte_ejecutivo)

print(reporte_ejecutivo)
print(f"\n✓ Reporte ejecutivo guardado: {REPORTS_DIR / 'reporte_ejecutivo_final.md'}")

---
## 9. RESUMEN FINAL DE ARCHIVOS GENERADOS

In [ ]:
print("\n" + "="*80)
print("RESUMEN DE TODOS LOS ARCHIVOS GENERADOS")
print("="*80)

print("\n📁 DATOS PROCESADOS (data/processed/):")
for archivo in DATA_PROCESSED_DIR.glob('*'):
    print(f"  ✓ {archivo.name}")

print("\n📊 RESULTADOS Y TABLAS (data/results/):")
for archivo in RESULTS_DIR.glob('*.csv'):
    print(f"  ✓ {archivo.name}")

print("\n📈 GRÁFICAS (outputs/graficas/):")
for archivo in OUTPUTS_DIR.glob('*.png'):
    print(f"  ✓ {archivo.name}")

print("\n📄 REPORTES (outputs/reportes/):")
for archivo in REPORTS_DIR.glob('*'):
    print(f"  ✓ {archivo.name}")

print("\n💾 SCRIPTS SQL (sql/):")
for archivo in SQL_DIR.glob('*.sql'):
    print(f"  ✓ {archivo.name}")

print("\n" + "="*80)
print("¡PIPELINE COMPLETO EJECUTADO CON ÉXITO!")
print("="*80)
print("\n✅ Datos listos para Power BI")
print("✅ Modelos predictivos entrenados")
print("✅ KPIs calculados")
print("✅ Documentación completa generada")
print("\nPróximo paso: Importar datos en Power BI usando la guía proporcionada")

---
**FIN DEL NOTEBOOK 5**

**PROYECTO COMPLETADO**

---

## Próximos Pasos:

1. **Importar datos en Power BI** siguiendo la guía en `outputs/reportes/guia_powerbi.txt`
2. **Crear visualizaciones** según las recomendaciones del reporte
3. **Validar proyecciones** con datos reales futuros
4. **Presentar resultados** a stakeholders universitarios

---

**¡Gracias por usar este pipeline de análisis!**